In [1]:
import pandas as pd
import glob
import os
from datetime import date
import numpy as np

### Acesso a pasta
PASTA = "/Users/franciscoabraao/Documents/Coleta_Varejo/coletas_precos_ipynb/precos_semana_14"

arquivos = glob.glob(os.path.join(PASTA, "precos_*.csv"))

print(f"Arquivos encontrados: {len(arquivos)}")

### Abrindo os arquivos 
dfs = []

for arq in arquivos:
    try:
        df = pd.read_csv(arq)
        df["arquivo_origem"] = os.path.basename(arq)  # ajuda debug
        dfs.append(df)
    except Exception as e:
        print(f"Erro ao ler {arq}: {e}")

### Juntando os arquivos em um unico df
df_raw = pd.concat(dfs, ignore_index=True, sort=False)
print("Shape inicial:", df_raw.shape)
print("Colunas encontradas:", df_raw.columns.tolist())

### Ajustando a data df = df_raw.copy()
df = df_raw.copy()
df["data"] = pd.to_datetime(
    df["data"],
    format="%d/%m/%Y",
    errors="coerce"
).dt.normalize()  # força 00:00:00

### Eliminando dados noneCOLUNAS_OBRIGATORIAS = ["data", "rede", "loja", "produto", "preco"]
COLUNAS_OBRIGATORIAS = ["data", "rede", "loja", "produto", "preco"]


antes = len(df)
df = df.dropna(subset=COLUNAS_OBRIGATORIAS)
depois = len(df)
print(f"Linhas antes: {antes}")
print(f"Linhas depois: {depois}")
print(f"Removidas: {antes - depois}")

#### Arquivo pronto para inclusao no db.precos_diarios
df_insert = df[["data", "rede", "loja", "produto", "preco", "desconto"]].copy()
df_insert.info()

### Retirar linhas duplicadas
antes = len(df_insert)

df_insert = df_insert.drop_duplicates(
    subset=["data", "rede", "loja", "produto"],
    keep="last"
)

depois = len(df_insert)

print("Linhas antes:", antes)
print("Linhas depois:", depois)
print("Removidas:", antes - depois)



Arquivos encontrados: 180
Shape inicial: (161382, 7)
Colunas encontradas: ['data', 'rede', 'loja', 'produto', 'preco', 'desconto', 'arquivo_origem']
Linhas antes: 161382
Linhas depois: 161382
Removidas: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 161382 entries, 0 to 161381
Data columns (total 6 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   data      161382 non-null  datetime64[ns]
 1   rede      161382 non-null  object        
 2   loja      161382 non-null  object        
 3   produto   161382 non-null  object        
 4   preco     161382 non-null  float64       
 5   desconto  46220 non-null   float64       
dtypes: datetime64[ns](1), float64(2), object(3)
memory usage: 7.4+ MB
Linhas antes: 161382
Linhas depois: 159183
Removidas: 2199


In [2]:
df_insert.data.max()

Timestamp('2026-04-01 00:00:00')

In [3]:
precos_do_dia = df_insert[df_insert["data"]==df_insert.data.max()]

In [4]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text 

import glob
import os
import re

import numpy as np

# Configurações de conexão
user = "franciscoabraao"
password = "Clarice2006"  # Adicione a senha se necessário
host = "localhost"
port = "5432"
# database = "painel_comex"

engine_precos = create_engine(
    f"postgresql://{user}:{password}@{host}:{port}/precos_varejo"
)

In [5]:
query = text("""
SELECT rede, loja, cidade, uf, cidade_uf, regiao, endereco, lat, "long"
FROM public.lojas;
""")

with engine_precos.connect() as conn:
    df_lojas = pd.read_sql(query, conn)

In [6]:
geo_all = df_lojas.copy()

In [7]:
precos_do_dia_geo = precos_do_dia.merge(
    geo_all[['loja', 'cidade', 'uf', 'regiao', 'lat', 'long']],
    on='loja',
    how='left'   # use left se quiser manter todas as linhas do df_semana_atual_2
)

In [8]:
precos_do_dia_geo.to_csv("precos_do_dia.csv", index=False)
